<a href="https://colab.research.google.com/github/Diego-Rodriguez16/mini-robots-2026-II/blob/main/7_IA_generativa/Taller_7_11_IA_Generativa_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller 7.11 — IA Generativa
Cuaderno para Google Colab. **Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*.

Contenido:
1. Cuatro skills (`SKILL.md`)
2. RAG industrial (chatbot con manuales técnicos)
3. Chatbot con los documentos del curso
4. Agente de mantenimiento
5. ¿LLM en un ESP32?

Ejecuta las celdas en orden.

## Configuración inicial (Google Drive)
Monta tu Drive. Todo se guarda en la carpeta `MyDrive/taller_ia`, así no tienes que subir ni descargar nada cada vez.
Colab te pedirá permiso **una vez por sesión**.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/taller_ia"
import os
os.makedirs(BASE, exist_ok=True)
print("Carpeta de trabajo:", BASE)

## Ejercicio 1 — Skills
Crea las 4 carpetas con su `SKILL.md` dentro de `taller_ia/skills` en tu Drive.

In [ ]:
SKILLS = {'documentar-ticket': '---\nname: documentar-ticket\ndescription: Documenta un ticket de trabajo con toda la descripción de lo que hay por hacer. Úsalo cuando el usuario pida crear, escribir o detallar un ticket o tarea.\n---\n\n# Documentar ticket\n\n## Pasos\n1. Si falta información, pregunta máximo 3 cosas (objetivo, alcance, restricciones).\n2. Escribe el ticket con la plantilla de abajo.\n3. Guarda el resultado en `ticket.md`.\n\n## Plantilla\n**Título:** verbo + objeto (ej. "Agregar login con correo")\n**Contexto:** por qué se necesita (2-3 líneas)\n**Objetivo:** qué debe lograrse\n**Alcance:** qué incluye / qué NO incluye\n**Tareas:** lista de pasos concretos\n**Criterios de aceptación:** lista verificable ("Dado... cuando... entonces...")\n**Riesgos / dependencias:** si los hay\n**Estimación:** S / M / L\n\n## Reglas\n- Lenguaje claro, sin ambigüedades.\n- Cada criterio de aceptación debe poder probarse.\n', 'implementar': '---\nname: implementar\ndescription: Implementa un ticket generando el código y sus pruebas. Úsalo cuando el usuario pida programar, implementar o codificar una tarea o ticket.\n---\n\n# Implementar\n\n## Pasos\n1. Lee el ticket (`ticket.md`) y confirma los criterios de aceptación.\n2. Escribe primero las pruebas (una por criterio de aceptación).\n3. Escribe el código mínimo para que las pruebas pasen.\n4. Ejecuta las pruebas y corrige hasta que todas pasen.\n5. Resume qué archivos creaste o cambiaste.\n\n## Reglas\n- Código simple y comentado en lo importante.\n- No agregues funciones que el ticket no pide.\n- Si una prueba falla, muestra el error y la corrección.\n', 'explicar-html': '---\nname: explicar-html\ndescription: Genera un archivo HTML que explica en detalle el trabajo realizado. Úsalo cuando el usuario pida una explicación, informe o resumen visual de lo implementado.\n---\n\n# Explicar en HTML\n\n## Pasos\n1. Revisa el ticket, el código y las pruebas.\n2. Genera un único archivo `explicacion.html` (CSS dentro del mismo archivo).\n3. Incluye estas secciones:\n   - Resumen (qué se hizo y por qué)\n   - Cómo funciona (paso a paso, con fragmentos de código en `<pre>`)\n   - Archivos modificados (tabla)\n   - Pruebas y resultados\n   - Limitaciones y próximos pasos\n\n## Reglas\n- Sin librerías externas: debe abrirse directo en el navegador.\n- Explica para alguien que no vio el código.\n', 'capturar-requerimientos': '---\nname: capturar-requerimientos\ndescription: Lee documentos provistos (PDF, Word, texto) y extrae los requerimientos. Úsalo cuando el usuario entregue documentos y pida identificar qué debe hacer el sistema.\n---\n\n# Capturar requerimientos\n\n## Pasos\n1. Lee completamente cada documento entregado.\n2. Extrae los requerimientos y clasifícalos:\n   - Funcionales (RF-01, RF-02...)\n   - No funcionales (RNF-01...): rendimiento, seguridad, etc.\n   - Restricciones\n3. Entrega una tabla: ID | Descripción | Fuente (documento y sección) | Prioridad.\n4. Lista aparte las ambigüedades o contradicciones encontradas.\n5. Guarda el resultado en `requerimientos.md`.\n\n## Reglas\n- No inventes requerimientos: todo debe tener una fuente.\n- Usa las palabras del documento cuando sea posible.\n'}

for nombre, texto in SKILLS.items():
    os.makedirs(f"{BASE}/skills/{nombre}", exist_ok=True)
    open(f"{BASE}/skills/{nombre}/SKILL.md", "w", encoding="utf-8").write(texto)

print(*sorted(os.listdir(f"{BASE}/skills")), sep="\n")

## Configuración de Ollama (para los ejercicios 2, 3 y 4)
Los modelos se guardan en tu Drive (`taller_ia/modelos`), así que solo se descargan **la primera vez** (~5 GB, necesitas ese espacio libre en Drive). Las siguientes veces solo se reinstala Ollama (1-2 min).

In [ ]:
!apt-get install -y zstd pciutils lshw -qq
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama pypdf numpy

In [ ]:
import subprocess, time
os.makedirs(f"{BASE}/modelos", exist_ok=True)
env = {**os.environ, "OLLAMA_MODELS": f"{BASE}/modelos"}
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
!ollama pull llama3.1:8b
!ollama pull nomic-embed-text

## Ejercicios 2 y 3 — RAG (chatbot con documentos)
El mismo código sirve para ambos; solo cambian los PDF:
- **Ejercicio 2 (industrial):** pon manuales técnicos de máquinas (bombas, motores, compresores...) en `MyDrive/taller_ia/manuales`. Los encuentras en sitios de fabricantes como Siemens, ABB, WEG o Grundfos.
- **Ejercicio 3 (curso):** pon los documentos del curso (por ejemplo, el PDF de IA Generativa) en `MyDrive/taller_ia/curso`.

Los PDF los copias a Drive **una sola vez**. Cambia `CARPETA` para alternar entre ejercicios. Opcional: si los manuales tienen enlace directo, ponlos en `URLS` y se descargan solos a esa carpeta.

Sigue los pasos de la sección 7.7: extraer texto → chunks → embeddings → base vectorial → búsqueda por similitud de coseno → respuesta del LLM.

In [ ]:
import urllib.request

CARPETA = "manuales"      # "manuales" (ejercicio 2)  o  "curso" (ejercicio 3)
URLS = []                 # opcional: enlaces directos a PDF, ej. ["https://sitio.com/manual.pdf"]

DOCS = f"{BASE}/{CARPETA}"
os.makedirs(DOCS, exist_ok=True)
for u in URLS:
    destino = f"{DOCS}/{u.split('/')[-1].split('?')[0]}"
    if not os.path.exists(destino):
        urllib.request.urlretrieve(u, destino)

pdfs = [f for f in os.listdir(DOCS) if f.lower().endswith(".pdf")]
print("PDF encontrados:", pdfs)
assert pdfs, f'Copia tus PDF a {DOCS} (en Drive) y vuelve a ejecutar esta celda'

In [ ]:
import numpy as np, ollama
from pypdf import PdfReader

EMB = "nomic-embed-text"
LLM = "llama3.1:8b"

def trocear(texto, tam=800, solape=100):
    return [texto[i:i+tam] for i in range(0, len(texto), tam - solape) if texto[i:i+tam].strip()]

chunks, fuentes = [], []
for f in sorted(os.listdir(DOCS)):
    if f.lower().endswith(".pdf"):
        for n, pag in enumerate(PdfReader(f"{DOCS}/{f}").pages, 1):
            for c in trocear(pag.extract_text() or ""):
                chunks.append(c)
                fuentes.append(f"{f} (pág. {n})")

print("Chunks:", len(chunks))
assert chunks, "No se encontró texto. Usa PDF con texto (no escaneados)."

vectores = []
for i in range(0, len(chunks), 32):         # por lotes
    vectores += ollama.embed(model=EMB, input=chunks[i:i+32]).embeddings
vectores = np.array(vectores)
vectores /= np.linalg.norm(vectores, axis=1, keepdims=True)
print("Base vectorial lista:", vectores.shape)

In [ ]:
def preguntar(pregunta, k=4):
    q = np.array(ollama.embed(model=EMB, input=pregunta).embeddings[0])
    q /= np.linalg.norm(q)
    top = np.argsort(vectores @ q)[::-1][:k]          # similitud de coseno
    contexto = "\n\n".join(f"[{fuentes[i]}]\n{chunks[i]}" for i in top)
    prompt = f"""Responde SOLO con el contexto. Si la respuesta no está, di "No lo encuentro en los documentos".
Cita la fuente entre corchetes.

CONTEXTO:
{contexto}

PREGUNTA: {pregunta}"""
    r = ollama.chat(model=LLM, messages=[{"role": "user", "content": prompt}])
    return r.message.content

PREGUNTA = "¿Qué es un modelo fundamental?"   # cambia la pregunta según tus documentos
print("Bot:", preguntar(PREGUNTA))

CHAT_INTERACTIVO = False   # ponlo en True para chatear (esta celda queda esperando tu entrada)
while CHAT_INTERACTIVO:
    p = input("Tú (escribe 'salir'): ")
    if p.strip().lower() == "salir":
        break
    print("Bot:", preguntar(p), "\n")

## Ejercicio 4 — Agente de mantenimiento
El LLM decide qué herramienta llamar; las herramientas son funciones de Python con datos de ejemplo:
buscar repuesto → consultar almacén → generar orden de trabajo → actualizar historial de fallas.

In [ ]:
import json
from datetime import datetime

# --- Datos de ejemplo ---
REPUESTOS = {"rodamiento 6205": "REP-001", "correa v": "REP-002",
             "sello mecanico": "REP-003", "filtro de aceite": "REP-004"}
ALMACEN = {"REP-001": 5, "REP-002": 0, "REP-003": 2, "REP-004": 10}
HISTORIAL = {}   # equipo -> lista de fallas
ORDENES = []

# --- Herramientas ---
def buscar_repuesto(descripcion: str) -> str:
    """Busca el código de un repuesto a partir de su descripción.

    Args:
        descripcion: nombre o descripción del repuesto, por ejemplo 'rodamiento 6205'.
    """
    d = descripcion.lower()
    for nombre, codigo in REPUESTOS.items():
        if nombre in d or d in nombre:
            return json.dumps({"codigo": codigo, "nombre": nombre})
    return json.dumps({"error": "Repuesto no encontrado"})

def consultar_almacen(codigo: str) -> str:
    """Consulta cuántas unidades hay en el almacén de un repuesto.

    Args:
        codigo: código del repuesto, por ejemplo 'REP-001'.
    """
    if codigo in ALMACEN:
        return json.dumps({"codigo": codigo, "stock": ALMACEN[codigo]})
    return json.dumps({"error": "Código no existe"})

def generar_orden_trabajo(equipo: str, falla: str, codigo_repuesto: str) -> str:
    """Genera una orden de trabajo de mantenimiento.

    Args:
        equipo: nombre del equipo con la falla.
        falla: descripción de la falla (incluye si no hay stock del repuesto).
        codigo_repuesto: código del repuesto necesario, o 'N/A' si no se requiere.
    """
    orden = {"id": f"OT-{len(ORDENES)+1:03d}", "equipo": equipo, "falla": falla,
             "repuesto": codigo_repuesto, "fecha": datetime.now().strftime("%Y-%m-%d %H:%M")}
    ORDENES.append(orden)
    return json.dumps(orden)

def actualizar_historial(equipo: str, falla: str) -> str:
    """Registra una falla en el historial de fallas de un equipo.

    Args:
        equipo: nombre del equipo.
        falla: descripción de la falla.
    """
    HISTORIAL.setdefault(equipo, []).append(
        {"falla": falla, "fecha": datetime.now().strftime("%Y-%m-%d")})
    return json.dumps({"ok": True, "total_fallas": len(HISTORIAL[equipo])})

HERRAMIENTAS = [buscar_repuesto, consultar_almacen, generar_orden_trabajo, actualizar_historial]
DISPONIBLES = {f.__name__: f for f in HERRAMIENTAS}

# --- Agente ---
SISTEMA = """Eres un agente de mantenimiento industrial. Para una falla debes, en orden:
1) buscar el repuesto, 2) consultar el almacén, 3) generar la orden de trabajo
(si no hay stock, indícalo en la falla), 4) actualizar el historial de fallas.
Usa siempre las herramientas, no inventes códigos. Al final resume lo hecho en español."""

def agente(solicitud, max_pasos=10):
    msgs = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": solicitud}]
    for _ in range(max_pasos):
        r = ollama.chat(model="llama3.1:8b", messages=msgs, tools=HERRAMIENTAS)
        msgs.append(r.message)
        if not r.message.tool_calls:
            return r.message.content
        for c in r.message.tool_calls:
            nombre, args = c.function.name, dict(c.function.arguments)
            print(f"🔧 {nombre}({args})")
            try:
                salida = DISPONIBLES[nombre](**args)
            except Exception as e:
                salida = json.dumps({"error": str(e)})
            msgs.append({"role": "tool", "tool_name": nombre, "content": salida})
    return "El agente alcanzó el máximo de pasos."

In [ ]:
print(agente("La bomba P-101 hace ruido fuerte, creo que falla el rodamiento 6205."))
print("\n" + "-"*60 + "\n")
print(agente("El ventilador V-02 tiene la correa v rota."))
print("\nÓrdenes:", json.dumps(ORDENES, indent=2, ensure_ascii=False))
print("Historial:", json.dumps(HISTORIAL, indent=2, ensure_ascii=False))

En la segunda solicitud, `REP-002` tiene stock 0: el agente debe reportarlo en la orden de trabajo. Si el modelo se salta algún paso, vuelve a ejecutar la celda.

## Ejercicio 5 — ¿Un LLM en un ESP32?

**Respuesta corta:** un LLM "real" (como los de este documento) **no** puede correr en un ESP32, pero sí puede usarse de dos formas.

| Recurso | ESP32 típico | LLM pequeño (1B parámetros, 4 bits) |
|---|---|---|
| RAM | ~520 KB (algunos con 8 MB PSRAM) | ~500 MB |
| Flash | 4–16 MB | ~600 MB |
| CPU | 2 núcleos a 240 MHz | GPU o CPU potente |
| Energía | Milivatios | Decenas de vatios |

**Restricciones**
- **Memoria:** el modelo, la ventana de contexto y los embeddings no caben (faltan ~3 órdenes de magnitud).
- **Velocidad:** sin GPU y a 240 MHz, generaría un token en muchos segundos o minutos.
- **Cuantización obligatoria:** pesos de 4 u 8 bits, con pérdida de calidad.
- **Contexto muy corto:** no se puede hacer RAG ni mantener conversaciones largas.
- **Sin aceleración de IA:** no hay hardware dedicado para multiplicar matrices.

**Qué sí se puede hacer**
1. **Modelo diminuto en el chip (TinyML / TinyLLM):** por ejemplo un modelo tipo *llama2.c* de ~260 mil parámetros, o modelos de clasificación (palabras clave, detección de anomalías de vibración). Solo sirven para tareas muy específicas.
2. **ESP32 como cliente (la opción práctica):** lee sensores y envía la consulta por WiFi a un servidor con el LLM (Ollama en un PC o una API en la nube). Requiere conexión, tiene latencia y depende del servidor.

**Ejemplo en planta:** un ESP32 mide vibración y temperatura de un motor y envía los datos por WiFi al agente de mantenimiento del ejercicio 4, que genera la orden de trabajo.